[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/049.%20The%20Causal%20%26%20Regression-Based%20Forecasting%20Problem/P49-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 49. The Causal and Regression-Based Forecasting Problem

## Tier 4: An Ensemble-Based Forecasting Demonstration

This notebook builds a synthetic port-throughput regression example, fits three different model types, and combines their predictions with validation-based inverse-MSE weights. It also examines model disagreement as a rough indicator of predictive uncertainty.

### Goals and inputs

The predictors describe economic activity, port operations, weather, and a time coordinate. The target is synthetic port throughput, measured in TEU. The notebook generates the data from a specified nonlinear equation, trains models on a training subset, uses a validation subset to calculate ensemble weights, and reports performance on a final holdout subset.

### Modeling notes and scope

The three models are **not deep neural networks**. They are deliberately named by their roles in this demonstration: polynomial Ridge regression is the feedforward-style model, a Random Forest using lagged predictors is the temporal-style model, and RBF Support Vector Regression (SVR) is the attention-style comparison model. These are conventional machine-learning models, not implementations of a neural network, an LSTM, or an attention mechanism. Permutation importance measures the change in a chosen score when a feature is shuffled; it is not attention and does not establish causality.

The target data are synthetic, and the rows are ordered by a generated time coordinate. Random train, validation, and test subsets make this a model-comparison demonstration, not a strict prospective forecasting evaluation. Lagged predictors are formed in chronological row order before subset selection. This assumes those earlier predictor values would be available when making a prediction. The example does not model deployment constraints, changing data distributions, missing observations, or operational capacity decisions.

### Method outline

1. Generate a reproducible synthetic dataset with nonlinear and temporal components.
2. Split row indices into training and test subsets, then divide the test subset into validation and final test subsets.
3. Fit feature and target scaling on training data only.
4. Fit three distinct regressors, using polynomial features or chronological lags where appropriate.
5. Assign ensemble weights in proportion to each model's inverse validation MSE.
6. Compare predictions on the final test subset and describe ensemble disagreement without treating it as a calibrated prediction interval.

### Ensemble definition

Let the three models be indexed by \(m\in\{1,2,3\}\). For a feature vector \(x_i\), model \(m\) produces prediction \(\hat{y}_{im}\). On a validation set of size \(n_v\), its mean squared error is

$$
\operatorname{MSE}_m = \frac{1}{n_v}\sum_{i=1}^{n_v}(y_i-\hat{y}_{im})^2.
$$

The ensemble assigns a normalized inverse-MSE weight to each model and forms a weighted average:

$$
w_m = \frac{1/\operatorname{MSE}_m}{\sum_{j=1}^{3}1/\operatorname{MSE}_j},
\qquad
\hat{y}_{i,\mathrm{ens}} = \sum_{m=1}^{3}w_m\hat{y}_{im}.
$$

Thus, lower validation error gives a model more influence. The weights are nonnegative and sum to one, provided all validation MSE values are positive. This rule is a specified weighting heuristic, not a general solution to a constrained optimization problem and not a guarantee that the ensemble will outperform every member.

For each example, the notebook also calculates the unweighted standard deviation across the three model predictions:

$$
u_i = \sqrt{\frac{1}{3}\sum_{m=1}^{3}(\hat{y}_{im}-\bar{y}_i)^2},
\qquad
\bar{y}_i = \frac{1}{3}\sum_{m=1}^{3}\hat{y}_{im}.
$$

This quantity measures disagreement among the models. It is not a confidence interval, a prediction interval, or a calibrated estimate of the probability of an error.

### Reading the results

MSE measures the average squared prediction error, so smaller values are preferable and the unit is squared TEU. \(R^2\) compares prediction error with variation around the target mean; a higher value indicates a better fit relative to that baseline, but does not by itself establish operational suitability. MAE is the average absolute error in TEU. Validation weights describe how the chosen rule distributes influence among the models; test metrics assess performance on the final holdout subset.

In [ ]:
# Import the libraries used for data generation, modeling, evaluation, and plots.
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
# Generate a reproducible synthetic dataset with nonlinear relationships.
np.random.seed(42)

n_samples = 150

trade_volume = np.random.normal(100, 20, n_samples)
gdp_growth = np.random.normal(2.5, 1.2, n_samples)
fuel_price = np.random.normal(85, 15, n_samples)
seasonal_peak = np.random.binomial(1, 0.35, n_samples)
ship_density = np.random.normal(25, 8, n_samples)
exchange_rate = np.random.normal(1.2, 0.2, n_samples)
competitor_activity = np.random.normal(50, 12, n_samples)
weather_severity = np.random.normal(0, 1, n_samples)

time_trend = np.linspace(0, 4 * np.pi, n_samples)
seasonal_effect = 50 * np.sin(time_trend) + 30 * np.cos(2 * time_trend)

true_throughput = (
    2000
    + 12 * trade_volume
    + 280 * gdp_growth
    + 6 * fuel_price
    + 220 * seasonal_peak
    + 18 * ship_density
    + 130 * exchange_rate
    - 15 * competitor_activity
    + 0.05 * trade_volume**2
    + 50 * np.sqrt(np.abs(gdp_growth)) * np.sign(gdp_growth)
    + 80 * (fuel_price - 80) * (fuel_price > 90)
    + 100 * trade_volume * gdp_growth / 100
    + 60 * seasonal_peak * ship_density / 25
    + seasonal_effect
    + 40 * weather_severity * (weather_severity > 1.5)
    + np.random.normal(0, 180, n_samples)
)

data = pd.DataFrame(
    {
        "trade_volume": trade_volume,
        "gdp_growth": gdp_growth,
        "fuel_price": fuel_price,
        "seasonal_peak": seasonal_peak,
        "ship_density": ship_density,
        "exchange_rate": exchange_rate,
        "competitor_activity": competitor_activity,
        "weather_severity": weather_severity,
        "time_trend": time_trend,
        "throughput": true_throughput,
    }
)

print("Synthetic Port Throughput Dataset")
print(f"Shape: {data.shape}")
print(f"Number of predictors: {data.shape[1] - 1}")
print("\nDataset statistics:")
print(data.describe().round(2))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Predictor and Throughput Relationships", fontsize=16, fontweight="bold")

predictors_to_plot = [
    "trade_volume",
    "gdp_growth",
    "fuel_price",
    "seasonal_peak",
    "ship_density",
    "time_trend",
]
for i, predictor in enumerate(predictors_to_plot):
    row, col = divmod(i, 3)
    axes[row, col].scatter(data[predictor], data["throughput"], alpha=0.6, s=30)
    axes[row, col].set_xlabel(predictor.replace("_", " ").title())
    axes[row, col].set_ylabel("Throughput (TEU)")
    axes[row, col].set_title(f"Throughput vs. {predictor.replace('_', ' ').title()}")
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 1. Prepare predictors and chronological features

A model can learn from predictors only after they have been represented numerically. Standardization expresses each variable in units of its training-set standard deviation; it helps models whose fitting depends on feature scale. The scaler must be fitted on training data alone so that information from the held-out observations does not influence preprocessing.

The temporal model below uses lagged predictor columns. A lag of one at row \(t\) is the predictor vector from row \(t-1\). These columns are created in the original chronological order before random subsets are selected. In contrast, a shuffled subset is not itself a time series, so lags must not be generated by rolling within that subset.

A sequence-building function is also included to make the input shape for sequence models explicit. It is an educational representation of consecutive predictor windows; the fitted temporal-style model in this notebook is a Random Forest with lagged columns, not an LSTM.

In [ ]:
predictor_columns = [column for column in data.columns if column != "throughput"]
X_raw = data[predictor_columns].to_numpy()
y_raw = data["throughput"].to_numpy()

# Split indices so that all predictors, targets, and derived features stay aligned.
all_indices = np.arange(len(data))
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.2,
    random_state=42,
)

scaler_X = StandardScaler()
scaler_X.fit(X_raw[train_indices])
X_scaled = scaler_X.transform(X_raw)

scaler_y = StandardScaler()
scaler_y.fit(y_raw[train_indices].reshape(-1, 1))
y_scaled = scaler_y.transform(y_raw.reshape(-1, 1)).ravel()

X_train = X_scaled[train_indices]
X_test = X_scaled[test_indices]
y_train = y_scaled[train_indices]
y_test = y_scaled[test_indices]
y_train_orig = y_raw[train_indices]
y_test_orig = y_raw[test_indices]

print("Data prepared for model fitting:")
print(f"Training samples: {len(train_indices)}")
print(f"Test samples before validation split: {len(test_indices)}")
print(f"Predictors: {X_train.shape[1]}")


def create_sequences(X, y, sequence_length=3):
    """Return consecutive predictor windows and their next-step targets."""
    if sequence_length < 1 or sequence_length >= len(X):
        raise ValueError("sequence_length must be positive and shorter than the data.")

    X_seq = []
    y_seq = []
    for i in range(sequence_length, len(X)):
        X_seq.append(X[i - sequence_length : i])
        y_seq.append(y[i])

    return np.asarray(X_seq), np.asarray(y_seq)


sequence_length = 3
X_sequences, y_sequences = create_sequences(X_scaled, y_scaled, sequence_length)
print("\nChronological sequence representation:")
print(f"Predictor sequence shape: {X_sequences.shape}")
print(f"Next-step target shape: {y_sequences.shape}")


def create_temporal_features(X, lags=2):
    """Append earlier chronological predictor rows as lagged columns."""
    if lags < 1 or lags >= len(X):
        raise ValueError("lags must be positive and shorter than the data.")

    temporal_columns = [X]
    for lag in range(1, lags + 1):
        lagged_features = np.empty_like(X)
        lagged_features[:lag] = X[0]
        lagged_features[lag:] = X[:-lag]
        temporal_columns.append(lagged_features)

    return np.column_stack(temporal_columns)


X_temporal_all = create_temporal_features(X_scaled, lags=2)
X_train_temporal = X_temporal_all[train_indices]
X_test_temporal = X_temporal_all[test_indices]

print(f"Temporal-style training feature shape: {X_train_temporal.shape}")
print(f"Temporal-style test feature shape: {X_test_temporal.shape}")

## 2. Fit the feedforward-style model

The feedforward-style model uses Ridge regression on the original standardized predictors and their squares. Squaring a feature allows the fitted relationship to curve, while Ridge regularization penalizes large coefficients and can reduce overfitting. This feature expansion does not include pairwise interactions; those are not added by this model.

If \(z\) is the standardized predictor vector, the model receives \([z_1,\ldots,z_p,z_1^2,\ldots,z_p^2]\). Ridge regression fits a linear function of this expanded representation, which corresponds to a restricted nonlinear function of the original predictors.

In [ ]:
print("Training feedforward-style model: Ridge regression with squared features")
print("=" * 72)

ff_model = Ridge(alpha=1.0, random_state=42)
X_train_poly = np.column_stack([X_train, X_train**2])
X_test_poly = np.column_stack([X_test, X_test**2])
ff_model.fit(X_train_poly, y_train)

ff_train_pred = ff_model.predict(X_train_poly)
ff_test_pred = ff_model.predict(X_test_poly)
ff_train_pred_orig = scaler_y.inverse_transform(ff_train_pred.reshape(-1, 1)).ravel()
ff_test_pred_orig = scaler_y.inverse_transform(ff_test_pred.reshape(-1, 1)).ravel()

ff_train_mse = mean_squared_error(y_train_orig, ff_train_pred_orig)
ff_test_mse = mean_squared_error(y_test_orig, ff_test_pred_orig)
ff_train_r2 = r2_score(y_train_orig, ff_train_pred_orig)
ff_test_r2 = r2_score(y_test_orig, ff_test_pred_orig)

print("\nFeedforward-style model results:")
print(f"Training MSE: {ff_train_mse:.2f} TEU²; R²: {ff_train_r2:.4f}")
print(f"Test-subset MSE: {ff_test_mse:.2f} TEU²; R²: {ff_test_r2:.4f}")
print(f"Expanded feature count: {X_train_poly.shape[1]}")

## 3. Fit the temporal-style model

A lagged feature gives the model information about earlier predictor values. The temporal-style model uses a Random Forest, whose tree splits can represent nonlinear interactions in the supplied columns. It does not have the recurrent state or gating mechanism of an LSTM, and the lagged inputs do not make it a neural sequence model.

The columns are derived from the full chronological predictor table and then selected by row index. This preserves the meaning of each lag even though the training and test subsets are randomly sampled.

In [ ]:
print("\nTraining temporal-style model: Random Forest with lagged predictors")
print("=" * 72)

lstm_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    max_depth=10,
)
lstm_model.fit(X_train_temporal, y_train)

lstm_train_pred = lstm_model.predict(X_train_temporal)
lstm_test_pred = lstm_model.predict(X_test_temporal)
lstm_train_pred_orig = scaler_y.inverse_transform(lstm_train_pred.reshape(-1, 1)).ravel()
lstm_test_pred_orig = scaler_y.inverse_transform(lstm_test_pred.reshape(-1, 1)).ravel()

lstm_train_mse = mean_squared_error(y_train_orig, lstm_train_pred_orig)
lstm_test_mse = mean_squared_error(y_test_orig, lstm_test_pred_orig)
lstm_train_r2 = r2_score(y_train_orig, lstm_train_pred_orig)
lstm_test_r2 = r2_score(y_test_orig, lstm_test_pred_orig)

print("\nTemporal-style model results:")
print(f"Training MSE: {lstm_train_mse:.2f} TEU²; R²: {lstm_train_r2:.4f}")
print(f"Test-subset MSE: {lstm_test_mse:.2f} TEU²; R²: {lstm_test_r2:.4f}")
print(f"Predictor columns including lags: {X_train_temporal.shape[1]}")

## 4. Fit the attention-style comparison model

The third model is an RBF-kernel SVR. Its kernel compares observations through distances in standardized predictor space and can represent nonlinear relationships. It does not use attention weights. To provide a separate feature diagnostic, permutation importance measures how much the model's validation-independent test score changes when each predictor is shuffled. A large decrease in score suggests that the fitted model relies on that predictor in this dataset; it is not a causal effect or a measure of attention.

In [ ]:
print("\nTraining attention-style comparison: RBF Support Vector Regression")
print("=" * 72)

attention_model = SVR(kernel="rbf", C=100, gamma=0.1, epsilon=0.1)
attention_model.fit(X_train, y_train)

attention_train_pred = attention_model.predict(X_train)
attention_test_pred = attention_model.predict(X_test)
attention_train_pred_orig = scaler_y.inverse_transform(
    attention_train_pred.reshape(-1, 1)
).ravel()
attention_test_pred_orig = scaler_y.inverse_transform(
    attention_test_pred.reshape(-1, 1)
).ravel()

attention_train_mse = mean_squared_error(y_train_orig, attention_train_pred_orig)
attention_test_mse = mean_squared_error(y_test_orig, attention_test_pred_orig)
attention_train_r2 = r2_score(y_train_orig, attention_train_pred_orig)
attention_test_r2 = r2_score(y_test_orig, attention_test_pred_orig)

print("\nRBF SVR results:")
print(f"Training MSE: {attention_train_mse:.2f} TEU²; R²: {attention_train_r2:.4f}")
print(f"Test-subset MSE: {attention_test_mse:.2f} TEU²; R²: {attention_test_r2:.4f}")
print(f"Predictors used: {X_train.shape[1]}")

perm_importance = permutation_importance(
    attention_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_squared_error",
)
feature_importance = sorted(
    zip(predictor_columns, perm_importance.importances_mean),
    key=lambda item: item[1],
    reverse=True,
)

print("\nPermutation importance on the test subset (increase in MSE):")
for rank, (feature, importance) in enumerate(feature_importance[:5], start=1):
    print(f"  {rank}. {feature}: {importance:.4f}")

## 5. Calculate ensemble weights and evaluate the final holdout

The ensemble retains each fitted model and combines their predictions. First, the held-out portion is divided into validation and final-test subsets. Validation predictions determine the inverse-MSE weights; the final test subset is reserved for the reported ensemble evaluation.

The weight calculation uses standardized target values. Because all three models use the same target scaling, multiplying all MSE values by the same positive constant would leave the normalized inverse-MSE weights unchanged. The final evaluation converts predictions back to TEU so that errors have a useful physical interpretation.

The standard deviation across the three predictions is reported as model disagreement. It describes variation among these particular fitted models and should not be interpreted as a calibrated uncertainty interval.

In [ ]:
class DeepEnsemble:
    """Combine the predictions of the three fitted model types."""

    def __init__(self, models, model_types):
        if len(models) != len(model_types) or not models:
            raise ValueError("models and model_types must have the same nonzero length.")

        self.models = models
        self.model_types = model_types
        self.n_models = len(models)
        self.ensemble_weights = np.ones(self.n_models) / self.n_models

    def predict(self, X, X_poly=None, X_temporal=None):
        """Return the weighted prediction and each model's prediction."""
        predictions = []
        for model, model_type in zip(self.models, self.model_types):
            if model_type == "ff":
                if X_poly is None:
                    raise ValueError("X_poly is required for the feedforward-style model.")
                prediction = model.predict(X_poly)
            elif model_type == "lstm":
                if X_temporal is None:
                    raise ValueError("X_temporal is required for the temporal-style model.")
                prediction = model.predict(X_temporal)
            else:
                prediction = model.predict(X)
            predictions.append(prediction)

        predictions = np.asarray(predictions)
        ensemble_prediction = np.average(
            predictions,
            axis=0,
            weights=self.ensemble_weights,
        )
        return ensemble_prediction, predictions

    def calculate_optimal_weights(self, X, X_poly, X_temporal, y):
        """Set normalized inverse-validation-MSE weights."""
        _, individual_predictions = self.predict(X, X_poly, X_temporal)
        mses = np.asarray(
            [mean_squared_error(y, prediction) for prediction in individual_predictions],
            dtype=float,
        )
        if not np.all(np.isfinite(mses)) or np.any(mses <= 0):
            raise ValueError("Validation MSE values must be finite and strictly positive.")

        inverse_mses = 1.0 / mses
        self.ensemble_weights = inverse_mses / inverse_mses.sum()
        return self.ensemble_weights

    def predict_with_uncertainty(self, X, X_poly, X_temporal):
        """Return the ensemble prediction and between-model disagreement."""
        ensemble_prediction, individual_predictions = self.predict(
            X,
            X_poly,
            X_temporal,
        )
        disagreement = np.std(individual_predictions, axis=0)
        return ensemble_prediction, disagreement


models = [ff_model, lstm_model, attention_model]
model_types = ["ff", "lstm", "attention"]
model_names = ["Feedforward-style", "Temporal-style", "RBF SVR"]
ensemble = DeepEnsemble(models, model_types)

# Keep row indices aligned when forming validation and final-test subsets.
validation_indices, final_test_indices = train_test_split(
    test_indices,
    test_size=0.5,
    random_state=42,
)

X_val = X_scaled[validation_indices]
y_val = y_scaled[validation_indices]
X_val_poly = np.column_stack([X_val, X_val**2])
X_val_temporal = X_temporal_all[validation_indices]

X_test_final = X_scaled[final_test_indices]
y_test_final = y_scaled[final_test_indices]
y_test_final_orig = y_raw[final_test_indices]
X_test_final_poly = np.column_stack([X_test_final, X_test_final**2])
X_test_final_temporal = X_temporal_all[final_test_indices]

optimal_weights = ensemble.calculate_optimal_weights(
    X_val,
    X_val_poly,
    X_val_temporal,
    y_val,
)

print("\n" + "=" * 64)
print("ENSEMBLE RESULTS")
print("=" * 64)
print("Validation-based inverse-MSE weights:")
for name, weight in zip(model_names, optimal_weights):
    print(f"  {name}: {weight:.4f}")

ensemble_train_pred, _ = ensemble.predict(
    X_train,
    X_train_poly,
    X_train_temporal,
)
ensemble_test_pred, ensemble_test_individual = ensemble.predict(
    X_test_final,
    X_test_final_poly,
    X_test_final_temporal,
)
ensemble_test_pred_unc, ensemble_uncertainty = ensemble.predict_with_uncertainty(
    X_test_final,
    X_test_final_poly,
    X_test_final_temporal,
)

ensemble_train_pred_orig = scaler_y.inverse_transform(
    ensemble_train_pred.reshape(-1, 1)
).ravel()
ensemble_test_pred_orig = scaler_y.inverse_transform(
    ensemble_test_pred.reshape(-1, 1)
).ravel()
ensemble_uncertainty_orig = ensemble_uncertainty * scaler_y.scale_[0]

ensemble_train_mse = mean_squared_error(y_train_orig, ensemble_train_pred_orig)
ensemble_test_mse = mean_squared_error(y_test_final_orig, ensemble_test_pred_orig)
ensemble_train_r2 = r2_score(y_train_orig, ensemble_train_pred_orig)
ensemble_test_r2 = r2_score(y_test_final_orig, ensemble_test_pred_orig)
ensemble_mae = mean_absolute_error(y_test_final_orig, ensemble_test_pred_orig)

individual_test_predictions_scaled = ensemble_test_individual
individual_test_predictions_orig = scaler_y.inverse_transform(
    individual_test_predictions_scaled.T
).T
individual_test_mse = [
    mean_squared_error(y_test_final_orig, prediction)
    for prediction in individual_test_predictions_orig
]
individual_test_r2 = [
    r2_score(y_test_final_orig, prediction)
    for prediction in individual_test_predictions_orig
]

print("\nTraining-set ensemble performance (in-sample):")
print(f"MSE: {ensemble_train_mse:.2f} TEU²; R²: {ensemble_train_r2:.4f}")
print("Final test-set ensemble performance:")
print(f"MSE: {ensemble_test_mse:.2f} TEU²; R²: {ensemble_test_r2:.4f}")
print(f"MAE: {ensemble_mae:.2f} TEU")
print(
    "Mean between-model disagreement: "
    f"{np.mean(ensemble_uncertainty_orig):.2f} TEU"
)

print("\nFinal test-set model comparison:")
for name, mse, score in zip(model_names, individual_test_mse, individual_test_r2):
    print(f"{name}: MSE={mse:.2f} TEU²; R²={score:.4f}")
print(f"Ensemble: MSE={ensemble_test_mse:.2f} TEU²; R²={ensemble_test_r2:.4f}")

best_individual_mse = min(individual_test_mse)
improvement = (best_individual_mse - ensemble_test_mse) / best_individual_mse * 100
print("\nEnsemble comparison with the best individual test MSE:")
print(f"Relative MSE change: {improvement:.2f}%")
if improvement > 0:
    print("The ensemble has lower test MSE than the best individual model in this split.")
elif improvement < 0:
    print("The ensemble has higher test MSE than the best individual model in this split.")
else:
    print("The ensemble and best individual model have equal test MSE in this split.")

## 6. Visualize performance, weights, predictions, and feature importance

The plots use the final test subset for model-performance and prediction comparisons. The uncertainty band shows one standard deviation of disagreement among the individual model predictions; it is a descriptive visualization, not a probability interval. Permutation importance is shown separately and is labeled as a score-based feature diagnostic.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Ensemble Forecasting Analysis", fontsize=16, fontweight="bold")

# Compare error and fit metrics on the same final test subset.
comparison_names = model_names + ["Ensemble"]
comparison_mse = individual_test_mse + [ensemble_test_mse]
comparison_r2 = individual_test_r2 + [ensemble_test_r2]
x = np.arange(len(comparison_names))
width = 0.35

axes[0, 0].bar(
    x - width / 2,
    comparison_mse,
    width,
    label="MSE (TEU²)",
    color="orange",
    alpha=0.75,
)
right_axis = axes[0, 0].twinx()
right_axis.bar(
    x + width / 2,
    comparison_r2,
    width,
    label="R-squared",
    color="blue",
    alpha=0.65,
)
axes[0, 0].set_xlabel("Model")
axes[0, 0].set_ylabel("Mean squared error (TEU²)", color="darkorange")
right_axis.set_ylabel("R-squared", color="blue")
axes[0, 0].set_title("Final Test-Set Performance")
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(comparison_names, rotation=30, ha="right")
axes[0, 0].grid(True, axis="y", alpha=0.3)
left_handles, left_labels = axes[0, 0].get_legend_handles_labels()
right_handles, right_labels = right_axis.get_legend_handles_labels()
axes[0, 0].legend(
    left_handles + right_handles,
    left_labels + right_labels,
    loc="best",
)

axes[0, 1].bar(
    model_names,
    optimal_weights,
    color=["skyblue", "lightcoral", "lightgreen"],
    alpha=0.85,
    edgecolor="black",
)
axes[0, 1].set_xlabel("Model")
axes[0, 1].set_ylabel("Validation-based ensemble weight")
axes[0, 1].set_title("Inverse-MSE Ensemble Weights")
axes[0, 1].tick_params(axis="x", rotation=25)
axes[0, 1].grid(True, axis="y", alpha=0.3)
for position, weight in enumerate(optimal_weights):
    axes[0, 1].text(position, weight, f"{weight:.3f}", ha="center", va="bottom")

sample_count = min(30, len(y_test_final_orig))
sample_indices = np.arange(sample_count)
y_sample = y_test_final_orig[:sample_count]
prediction_sample = ensemble_test_pred_orig[:sample_count]
disagreement_sample = ensemble_uncertainty_orig[:sample_count]

axes[1, 0].plot(
    sample_indices,
    y_sample,
    "o-",
    label="Observed throughput",
    linewidth=2,
    markersize=5,
)
axes[1, 0].plot(
    sample_indices,
    prediction_sample,
    "s--",
    label="Ensemble prediction",
    linewidth=2,
    markersize=5,
)
axes[1, 0].fill_between(
    sample_indices,
    prediction_sample - disagreement_sample,
    prediction_sample + disagreement_sample,
    alpha=0.25,
    label="One standard deviation of model disagreement",
)
axes[1, 0].set_xlabel("Final test-set row position")
axes[1, 0].set_ylabel("Throughput (TEU)")
axes[1, 0].set_title("Predictions and Between-Model Disagreement")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

importance_to_plot = feature_importance[:5]
feature_labels = [item[0].replace("_", " ").title() for item in importance_to_plot]
importance_values = [item[1] for item in importance_to_plot]
axes[1, 1].barh(feature_labels, importance_values, color="purple", alpha=0.7)
axes[1, 1].set_xlabel("Increase in test MSE after permutation (standardized target units²)")
axes[1, 1].set_title("Top Permutation-Importance Features for RBF SVR")
axes[1, 1].grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 64)
print("PERMUTATION-IMPORTANCE SUMMARY")
print("=" * 64)
print("Largest score increases when a predictor is shuffled:")
for rank, (feature, importance) in enumerate(feature_importance[:5], start=1):
    print(f"  {rank}. {feature}: {importance:.4f}")
print("\nSmallest score increases when a predictor is shuffled:")
for rank, (feature, importance) in enumerate(feature_importance[-3:], start=1):
    print(f"  {rank}. {feature}: {importance:.4f}")
print("These values describe model reliance in this sample; they are not causal effects.")

## 7. Interpret disagreement and capacity context cautiously

Model disagreement can help identify cases where the fitted models make substantially different predictions. The percentile threshold below is a descriptive flag within this final test subset, not a statistical confidence threshold. The capacity calculation uses observed test-set throughput to describe the synthetic sample; it is not a count of predicted capacity breaches.

A synthetic example and a small random holdout cannot establish that a model is reliable for operational decisions. Metrics, disagreement, and capacity context should be interpreted together with data quality, the evaluation design, and the consequences of forecast error.

In [ ]:
print("\n" + "=" * 64)
print("DISAGREEMENT AND CAPACITY CONTEXT")
print("=" * 64)

high_disagreement_threshold = np.percentile(ensemble_uncertainty_orig, 75)
high_disagreement_indices = np.where(
    ensemble_uncertainty_orig > high_disagreement_threshold
)[0]

print("\nBetween-model disagreement on the final test subset:")
print(f"Mean disagreement: {np.mean(ensemble_uncertainty_orig):.2f} TEU")
print(f"Standard deviation: {np.std(ensemble_uncertainty_orig):.2f} TEU")
print(f"75th-percentile threshold: {high_disagreement_threshold:.2f} TEU")
print(
    "Rows above the threshold: "
    f"{len(high_disagreement_indices)}/{len(ensemble_uncertainty_orig)} "
    f"({len(high_disagreement_indices) / len(ensemble_uncertainty_orig) * 100:.1f}%)"
)

port_capacity = 5000
observed_capacity_utilization = y_test_final_orig / port_capacity * 100
high_observed_utilization = np.where(observed_capacity_utilization > 85)[0]

print("\nObserved capacity context for the final test subset:")
print(
    "Observed rows above 85% of the illustrative capacity: "
    f"{len(high_observed_utilization)}/{len(observed_capacity_utilization)} "
    f"({len(high_observed_utilization) / len(observed_capacity_utilization) * 100:.1f}%)"
)
print(
    "Mean observed utilization: "
    f"{np.mean(observed_capacity_utilization):.1f}% of the illustrative capacity"
)

print("\nInterpretation notes:")
if len(high_disagreement_indices) > 0:
    print(
        f"- {len(high_disagreement_indices)} test rows exceed the descriptive "
        "disagreement threshold; inspect these cases before relying on the forecast."
    )
else:
    print("- No test rows exceed the descriptive disagreement threshold.")
print("- Model disagreement is not a calibrated measure of forecast risk.")
print("- Capacity utilization above the illustrative threshold is descriptive, not a decision rule.")
print(f"- Final test-set ensemble R²: {ensemble_test_r2:.4f}")
print(f"- Final test-set ensemble MAE: {ensemble_mae:.2f} TEU")

print("\nFinal ensemble summary:")
print("Model types: Ridge with squared features, Random Forest with lags, and RBF SVR")
print(f"Validation-based weights: {dict(zip(model_names, optimal_weights.round(3)))}")
print(
    "Final test-set performance: "
    f"MSE={ensemble_test_mse:.2f} TEU², "
    f"R²={ensemble_test_r2:.4f}, MAE={ensemble_mae:.2f} TEU"
)
print(
    "Mean between-model disagreement: "
    f"{np.mean(ensemble_uncertainty_orig):.2f} TEU"
)
print("=" * 64)

## 8. How this tier differs from a simpler regression workflow

### Potential advantages

- **Nonlinear representation:** squared features, tree splits, and an RBF kernel can represent relationships that a simple linear model cannot.
- **Different model biases:** models with different structures may make different errors, which creates an opportunity for a weighted ensemble to perform well.
- **Validation-based combination:** weights are calculated from held-out validation errors rather than chosen by inspecting final test performance.
- **Model disagreement:** differences among model predictions provide a transparent diagnostic for cases that merit closer review.

### Limitations

- **These are not deep-learning architectures:** the notebook uses conventional regressors as role-based examples; it does not train neural networks, LSTMs, or attention models.
- **Small synthetic sample:** 150 generated observations are not evidence that the workflow will generalize to a real port.
- **Temporal evaluation is limited:** random splitting does not test performance on a strictly later time period, and the lagged Random Forest is not a recurrent sequence model.
- **Heuristic ensemble weights:** inverse validation MSE is a simple rule. Validation estimates can be noisy, especially when the validation subset is small.
- **Disagreement is not calibrated uncertainty:** similar predictions can still be wrong, and differing predictions do not directly give a probability or coverage guarantee.
- **Feature importance is not causal attribution:** permutation scores describe dependence of this fitted model's score on the observed predictors.
- **No operational constraints:** the example does not model port capacity, staffing, service levels, or the cost of forecast errors.

### Appropriate interpretation

Use this notebook to understand a specific ensemble workflow: fit distinct regressors, compute inverse-validation-MSE weights, combine predictions, and compare results on a final holdout. Any operational use would require data and evaluation designs suited to the intended forecasting horizon, calibrated risk estimates, and explicit decision constraints.